# Varraste Loendur – mudeli treening

See notebook treenib varda otste detektori (YOLO11n), mõõdab selle **loendusviga** kuldsel testikomplektil ja ekspordib ONNX-faili, mis läheb rakenduse `public/` kausta.

**Kuidas kasutada:** Runtime → Change runtime type → **T4 GPU**. Täida jaotis 1 (seaded), siis Runtime → **Run all**. Treening kestab 200 fotoga ~20–40 min. Kõik tulemused salvestatakse Google Drive'i, nii et sessiooni katkemine ei kaota midagi.

**Datasetti oodatav struktuur (YOLO-formaat)** – Roboflow "YOLOv8/YOLO11" eksport annab täpselt selle:
```
dataset/
  data.yaml           # path, train, val, test, names: ['bar_end']
  train/images, train/labels
  valid/images, valid/labels
  test/images,  test/labels     # KULDNE TEST – ainult mõõtmiseks, kunagi treeninguks
```
Iga `labels/*.txt` rida: `0 cx cy w h` (0–1 skaalas). Kuldse testi tegelik arv = ridade arv label-failis.

## 1 · Seaded

In [ ]:
#@title Seaded { display-mode: "form" }
KATSE_NIMI   = "yolo11n_v01"   #@param {type:"string"}
MUDEL        = "yolo11n.pt"    #@param ["yolo11n.pt", "yolo11s.pt"]
EPOHHE       = 100             #@param {type:"integer"}
IMGSZ        = 1024            #@param [640, 800, 1024, 1280] {type:"raw"}
BATCH        = 8               #@param {type:"integer"}

# Kust datasett tuleb: "roboflow" | "zip" (Drive'is olev zip) | "kaust" (Drive'is juba lahti pakitud)
ANDMED_ALLIKAS = "zip"          #@param ["roboflow", "zip", "kaust"]
ZIP_TEE        = "/content/drive/MyDrive/varraste-loendur/dataset.zip"   #@param {type:"string"}
KAUST_TEE      = "/content/drive/MyDrive/varraste-loendur/dataset"       #@param {type:"string"}
ROBOFLOW_API_KEY  = ""          #@param {type:"string"}
ROBOFLOW_WORKSPACE = ""         #@param {type:"string"}
ROBOFLOW_PROJECT   = ""         #@param {type:"string"}
ROBOFLOW_VERSION   = 1          #@param {type:"integer"}

# Kuhu tulemused lähevad (Drive, et sessiooni katkemine ei kaotaks midagi)
TULEMUSED = "/content/drive/MyDrive/varraste-loendur/runs"
print("OK – seaded loetud")

## 2 · Paigaldus ja Drive

In [ ]:
!pip -q install "ultralytics>=8.3" onnx onnxslim onnxruntime
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, time, shutil, yaml
from pathlib import Path
os.makedirs(TULEMUSED, exist_ok=True)
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "PUUDUB – vali Runtime → T4 GPU")

## 3 · Datasett

In [ ]:
DATASET = Path("/content/dataset")
if ANDMED_ALLIKAS == "roboflow":
    !pip -q install roboflow
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    ds = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION).download("yolov11", location=str(DATASET))
elif ANDMED_ALLIKAS == "zip":
    shutil.rmtree(DATASET, ignore_errors=True); DATASET.mkdir()
    !unzip -q "{ZIP_TEE}" -d "{DATASET}"
    # kui zipis oli üks alamkaust, tõsta selle sisu üles
    sub = [p for p in DATASET.iterdir() if p.is_dir()]
    if not (DATASET/"data.yaml").exists() and len(sub) == 1 and (sub[0]/"data.yaml").exists():
        for p in sub[0].iterdir(): shutil.move(str(p), DATASET)
else:
    DATASET = Path(KAUST_TEE)

yaml_path = DATASET/"data.yaml"
assert yaml_path.exists(), f"data.yaml puudub: {yaml_path}"
cfg = yaml.safe_load(open(yaml_path))
# tee teed absoluutseks ja kontrolli
for k in ("train","val","test"):
    if k in cfg and not str(cfg[k]).startswith("/"):
        cfg[k] = str(DATASET/str(cfg[k]).replace("../",""))
cfg["path"] = str(DATASET)
yaml.safe_dump(cfg, open(yaml_path,"w"))

def loenda(split):
    d = Path(cfg.get(split, "")) if split in cfg else None
    if not d or not d.exists(): return 0, 0
    imgs = [p for p in d.glob("*") if p.suffix.lower() in (".jpg",".jpeg",".png")]
    lbl = Path(str(d).replace("images","labels"))
    otsi = sum(len(open(f).read().strip().splitlines()) for f in lbl.glob("*.txt")) if lbl.exists() else 0
    return len(imgs), otsi
for s in ("train","val","test"):
    n, o = loenda(s); print(f"{s:5s}: {n:4d} fotot, {o:6d} märgistatud otsa")
print("klassid:", cfg.get("names"))
assert loenda("test")[0] >= 10, "Kuldne test peab olema vähemalt 10 fotot – ilma selleta pole numbritel tähendust"

## 4 · Treening

Tulemused salvestuvad `TULEMUSED/KATSE_NIMI` alla Drive'i. Kui Colab katkestab sessiooni, jooksuta see lahter uuesti – jätkab viimasest salvestatud epohhist. Lõpetatud katset ei treenita uuesti (fail `VALMIS.txt`); uue katse jaoks muuda `KATSE_NIMI`.

Augmentatsioon: pööramine ja heledus jah; **peegeldamine ja mosaiik** on vaikimisi sees ja sobivad meile (varda otsad on sümmeetrilised). `close_mosaic=10` lülitab mosaiigi viimastel epohhidel välja, et mudel õpiks tervet kimpu.

In [ ]:
from ultralytics import YOLO
run_dir = Path(TULEMUSED)/KATSE_NIMI
last, best, done = run_dir/"weights"/"last.pt", run_dir/"weights"/"best.pt", run_dir/"VALMIS.txt"
TREENI = dict(data=str(yaml_path), epochs=EPOHHE, imgsz=IMGSZ, batch=BATCH,
              project=TULEMUSED, name=KATSE_NIMI, exist_ok=True,
              patience=30, close_mosaic=10, degrees=15, hsv_v=0.5,
              save_period=10,            # kaalud Drive'i iga 10 epohhi järel
              plots=True, verbose=False)
t0 = time.time()
if done.exists():
    print(f"Katse '{KATSE_NIMI}' on juba lõpetatud ({done.read_text().strip()}) – treeningut ei korrata, kasutan best.pt.")
    print("Uue treeningu jaoks muuda jaotises 1 KATSE_NIMI.")
    model = YOLO(str(best))
elif last.exists():
    print("Jätkan katkenud treeningut:", last)
    try:
        model = YOLO(str(last)); model.train(resume=True)
    except Exception as e:
        print("Jätkamine ei õnnestunud, alustan otsast:", e)
        model = YOLO(MUDEL); model.train(**TREENI)
    done.write_text(f"valmis {time.strftime('%Y-%m-%d %H:%M')}, {(time.time()-t0)/60:.0f} min")
else:
    model = YOLO(MUDEL); model.train(**TREENI)
    done.write_text(f"valmis {time.strftime('%Y-%m-%d %H:%M')}, {(time.time()-t0)/60:.0f} min")
print(f"Kulus {(time.time()-t0)/60:.0f} min")
best = run_dir/"weights"/"best.pt"; assert best.exists()

## 5 · Loendusviga kuldsel testil

Treeningtööriist näitab mAP-i; laole loeb, **mitu varrast läks valesti**. Mõõdame iga testfoto kohta `|ennustatud − tegelik|` ja otsime kindluse läve (`conf`), mille juures viga on väikseim – see lävi läheb rakendusse.

In [ ]:
import numpy as np
model = YOLO(str(best))
test_imgs = sorted([p for p in Path(cfg["test"]).glob("*") if p.suffix.lower() in (".jpg",".jpeg",".png")])
lbl_dir = Path(str(cfg["test"]).replace("images","labels"))
truth = {p.stem: len(open(lbl_dir/f"{p.stem}.txt").read().strip().splitlines()) if (lbl_dir/f"{p.stem}.txt").exists() else 0 for p in test_imgs}

# üks predict madala lävega, hiljem filtreerime läve järgi (kiirem kui iga läve jaoks uuesti)
res = model.predict([str(p) for p in test_imgs], imgsz=IMGSZ, conf=0.05, iou=0.5, max_det=1000, verbose=False)
scores = {p.stem: r.boxes.conf.cpu().numpy() for p, r in zip(test_imgs, res)}

def mõõda(conf):
    err = np.array([abs(int((scores[s] >= conf).sum()) - truth[s]) for s in truth])
    return dict(conf=conf, mae=err.mean(), täpselt=(err==0).mean(), pm2=(err<=2).mean(), max=err.max())

rida = [mõõda(c) for c in np.arange(0.10, 0.71, 0.05)]
parim = min(rida, key=lambda r: (r["mae"], -r["täpselt"]))
print(f"{'conf':>5} {'keskm.viga':>10} {'täpselt':>8} {'±2':>6} {'max':>4}")
for r in rida:
    m = " ◀ parim" if r is parim else ""
    print(f"{r['conf']:5.2f} {r['mae']:10.2f} {r['täpselt']:8.0%} {r['pm2']:6.0%} {r['max']:4d}{m}")
CONF = round(float(parim["conf"]), 2)
print(f"\nRakendusse läheb conf = {CONF}. Keskmine viga {parim['mae']:.2f} varrast, täpselt õigeid {parim['täpselt']:.0%}, ±2 sees {parim['pm2']:.0%}")
if parim["mae"] > 3: print("⚠ Viga üle 3 varda – plaani järgi enne pilooti rohkem/paremaid fotosid, mitte teist mudelit.")

### Fotode kaupa – kus mudel eksib

In [ ]:
import matplotlib.pyplot as plt, cv2
halvimad = sorted(truth, key=lambda s: -abs(int((scores[s]>=CONF).sum()) - truth[s]))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, stem in zip(axes.flat, halvimad):
    r = res[[p.stem for p in test_imgs].index(stem)]
    img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    for (x1,y1,x2,y2), c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        if c >= CONF: cv2.rectangle(img, (int(x1),int(y1)), (int(x2),int(y2)), (224,176,0), 4)
    n = int((scores[stem]>=CONF).sum())
    ax.imshow(img); ax.set_title(f"{stem}: ennustas {n}, tegelik {truth[stem]}", color="red" if n!=truth[stem] else "green"); ax.axis("off")
plt.tight_layout(); plt.show()

## 6 · Eksport ONNX (rakenduse jaoks)

Fikseeritud sisend `1×3×IMGSZ×IMGSZ`, opset 12 – täpselt see, mida testleht ja rakendus ootavad. Fail kopeeritakse Drive'i; sealt käsitsi repo `public/` kausta (või `git push` otse Colabist, kui repo on siin kloonitud).

In [ ]:
onnx_path = model.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
out = Path(TULEMUSED)/f"{KATSE_NIMI}.onnx"
shutil.copy(onnx_path, out)
print(f"ONNX: {out}  ({out.stat().st_size/1e6:.1f} MB)")

# kontroll: ONNX annab sama arvu kui .pt
import onnxruntime as ort
sess = ort.InferenceSession(str(out), providers=["CPUExecutionProvider"])
p = test_imgs[0]; im = cv2.imread(str(p)); h, w = im.shape[:2]; r = IMGSZ/max(h, w)
canvas = np.full((IMGSZ, IMGSZ, 3), 114, np.uint8); rs = cv2.resize(im, (int(w*r), int(h*r)))
canvas[(IMGSZ-rs.shape[0])//2:(IMGSZ-rs.shape[0])//2+rs.shape[0], (IMGSZ-rs.shape[1])//2:(IMGSZ-rs.shape[1])//2+rs.shape[1]] = rs
x = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).transpose(2,0,1)[None].astype(np.float32)/255
y = sess.run(None, {"images": x})[0]            # [1, 4+klasse, N]
n_onnx = int((y[0, 4:].max(0) >= CONF).sum())   # enne NMS-i, seega ≥ lõplik arv
print(f"{p.name}: ONNX kandidaate ≥{CONF}: {n_onnx} (enne NMS-i), .pt lõplik: {int((scores[p.stem]>=CONF).sum())}, tegelik: {truth[p.stem]}")
print("Kui ONNX kandidaate on palju rohkem kui .pt – see on normaalne, NMS teeb ülejäänu brauseris.")

## 7 · Katse kirja

In [ ]:
import csv, datetime
log = Path(TULEMUSED)/"katsed.csv"
uus = not log.exists()
with open(log, "a", newline="") as f:
    w = csv.writer(f)
    if uus: w.writerow(["kuupäev","katse","mudel","epohhe","imgsz","train_fotod","conf","keskm_viga","täpselt","pm2","max_viga","onnx"])
    w.writerow([datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), KATSE_NIMI, MUDEL, EPOHHE, IMGSZ, loenda("train")[0],
                CONF, f"{parim['mae']:.2f}", f"{parim['täpselt']:.2f}", f"{parim['pm2']:.2f}", parim["max"], out.name])
print(open(log).read())

## Järgmine katse

Muuda jaotises 1 `KATSE_NIMI` ja `MUDEL` (nt `yolo11s.pt`) ja jooksuta uuesti – sama datasett, sama test, uus rida `katsed.csv`-s. D-FINE-N jaoks on eraldi notebook (teine teek), aga testikomplekt ja mõõdik on samad.